Step 0 — Mount Drive and Restore Session

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/doctalk'
print(f'Project folder: {PROJECT_DIR}')

Step 1 — Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q groq
!pip install -q langchain-groq

print('All dependencies installed.')

Step 2 — Reload Embedding Model

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

print('Loading embedding model...')

embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print('Embedding model loaded.')

Step 3 — Reload FAISS Index

In [ ]:
from langchain_community.vectorstores import FAISS

FAISS_PATH = f'{PROJECT_DIR}/faiss_index'

vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print('FAISS index loaded.')

Step 4 — Connect to Groq

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name='llama-3.3-70b-versatile',
    temperature=0
)

print('Groq LLM loaded.')
print('Model: llama-3.3-70b-versatile')

Step 5 — Build the Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are a helpful assistant that answers questions based strictly on the provided context.
If the answer is not found in the context, say "I could not find an answer in the document."
Do not use any knowledge outside of the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template=template
)

print('Prompt template created.')

Step 6a — Base Retriever

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_chunks(docs):
    return '\n\n'.join([
        f'[Page {doc.metadata.get("page", "?") + 1}]\n{doc.page_content}'
        for doc in docs
    ])

retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

print('Base retriever created.')

Step 6b — Build RAG Chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


rag_chain = (
    {
        'context': RunnableLambda(lambda q: retriever.invoke(q)) | format_chunks,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain built.')

Step 7 — Ask a Question

In [ ]:
question = "What is the temperature on Mars?"

print(f'Question: {question}')
print(f'\nAnswer:')

answer = rag_chain.invoke(question)
print(answer)

Step 8 — Answer with Sources

In [ ]:
question = "Temperature on mars?"

# Get answer
answer = rag_chain.invoke(question)

# Get source chunks separately
source_docs = retriever.invoke(question)

print(f'Question: {question}')
print(f'\nAnswer: {answer}')
print(f'\nSources:')
for i, doc in enumerate(source_docs):
    print(f'  [{i+1}] Page {doc.metadata.get("page", "?") + 1}: {doc.page_content[:100]}...')